# Inne modele Normalizing Flow: Glow i FFJORD

> **Cel notebooka:** poznać dwa różne sposoby rozszerzenia idei flow. Glow wprowadza uczoną odwracalną transformację kanałów oraz ActNorm, a FFJORD interpretuje przepływ jako ciągłe równanie różniczkowe i śledzi zmianę gęstości przez dywergencję pola wektorowego.

Wspólny cel pozostaje ten sam: dokładna (lub kontrolowanie przybliżona) gęstość i odwracalne generowanie. Różni się parametryzacja transformacji i kompromis pomiędzy kosztem obliczeń, pamięcią oraz elastycznością.

**Plan**
1. Glow: ActNorm, invertible 1×1 convolution, affine coupling
2. Zwięzła implementacja elementów Glow w PyTorch i testy round-trip
3. FFJORD / continuous normalizing flow (CNF): twierdzenie o ciągłości
4. Dyskretyzowany CNF w 2D i estymacja log-gęstości
5. Porównanie rodzin i wskazówki praktyczne


## 1. Glow: lepsze mieszanie kanałów

RealNVP często używa ustalonej permutacji pomiędzy coupling layers. Glow zastępuje ją uczoną, odwracalną macierzą $W$ (dla obrazu jest to transformacja kanałów, czyli invertible 1×1 convolution). Dla tensora $h\in\mathbb R^{C\times H\times W}$ tę samą macierz stosujemy w każdej pozycji przestrzennej:
$$\log|\det J|=H\,W\,\log|\det W|.$$

Aby uniknąć kosztownego i niestabilnego wyznaczania dowolnego $W$, Glow parametryzuje $W$ przez rozkład LU. Wtedy wyznacznik zależy tylko od iloczynu diagonalnych elementów $U$. Niżej dla klarowności implementujemy macierz $W$ bez parametryzacji LU; to adekwatne dla małej liczby kanałów, ale dla większych modeli warto zastosować LU.

**ActNorm** jest kanałową afiniczną transformacją z uczonymi skalą i przesunięciem. Inicjalizuje się ją na pierwszym batchu tak, by aktywacje miały średnią zero i wariancję jeden; po inicjalizacji parametry są zwykłymi uczonymi wagami, co stabilizuje trening niezależnie od wielkości minibatcha.

Glow step zwykle składa się z ActNorm → invertible 1×1 convolution → affine coupling. Te trzy operacje: normalizują skalę, mieszają kanały, a następnie wprowadzają warunkowaną nieliniowość.


In [ ]:
import math
import torch
import torch.nn as nn

torch.manual_seed(5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ActNorm(nn.Module):
    """ActNorm dla NCHW. Inicjalizuje się jeden raz na pierwszym batchu."""
    def __init__(self, channels, eps=1e-6):
        super().__init__(); self.bias = nn.Parameter(torch.zeros(1,channels,1,1)); self.log_scale = nn.Parameter(torch.zeros(1,channels,1,1))
        self.register_buffer('initialized', torch.tensor(False)); self.eps=eps
    @torch.no_grad()
    def initialize(self, x):
        mean = x.mean(dim=(0,2,3), keepdim=True); std = x.std(dim=(0,2,3), keepdim=True).clamp_min(self.eps)
        self.bias.copy_(-mean); self.log_scale.copy_(-std.log()); self.initialized.fill_(True)
    def forward(self, x):
        if not bool(self.initialized): self.initialize(x)
        y = (x + self.bias) * self.log_scale.exp()
        ld = x.shape[2]*x.shape[3]*self.log_scale.sum()
        return y, ld.expand(x.shape[0])
    def inverse(self, y): return y * (-self.log_scale).exp() - self.bias

class Invertible1x1(nn.Module):
    """Mała wersja z bezpośrednimi wagami; determinant liczymy przez slogdet."""
    def __init__(self, channels):
        super().__init__()
        q, _ = torch.linalg.qr(torch.randn(channels,channels))
        self.weight = nn.Parameter(q)
    def forward(self, x):
        _, logabsdet = torch.linalg.slogdet(self.weight)
        y = torch.einsum('ij,njhw->nihw', self.weight, x)
        ld = x.shape[2]*x.shape[3]*logabsdet
        return y, ld.expand(x.shape[0])
    def inverse(self, y):
        w_inv = torch.linalg.inv(self.weight)
        return torch.einsum('ij,njhw->nihw', w_inv, y)

an = ActNorm(4).to(device); conv = Invertible1x1(4).to(device)
x = torch.randn(8,4,6,6,device=device)
y, ld1 = an(x); xr = an.inverse(y)
v, ld2 = conv(y); yr = conv.inverse(v)
print('ActNorm max round-trip error:', (x-xr).abs().max().item())
print('1x1 conv max round-trip error:', (y-yr).abs().max().item())
print('Batch logdet:', ld1[:2].detach(), ld2[:2].detach())

## 2. Affine coupling i wieloskalowe obrazy w Glow

Glow coupling zachowuje schemat RealNVP: część kanałów zostaje, a sieć przewiduje shift i scale pozostałych kanałów. Warstwy 1×1 convolution uczą miksowania kanałów, więc sztywne maski mogą być wystarczające. W obrazach Glow używa squeeze: układ przestrzenny 2×2 przekształca się w dodatkowe kanały. Potem część kanałów może zostać „wypuszczona” do latentów na kilku skalach. To redukuje obciążenie późniejszych warstw i tworzy hierarchiczną latentną reprezentację.

**Koszt:** logdet konwolucji skaluje się przez $H\cdot W$, a bez LU każda lokalizacja wymaga wyznacznika macierzy kanałów. Duże kanały zwiększają koszt; z LU logdet jest tani, lecz trzeba pilnować stabilności diagonalnych parametrów.

Powyższy kod implementuje dwa ważne klocki Glow, a nie kompletny image model. Pełny Glow wymaga sieci coupling, squeeze/split, wieloskalowego priory i obsługi dequantyzacji pikseli. Należy jasno rozróżniać demonstrację komponentu od kompletnego modelu.


## 3. FFJORD: przepływ jako ciągły czas

Discrete flow nakłada skończoną liczbę map. Continuous normalizing flow opisuje trajektorię $z(t)$ równaniem:
$$\frac{dz(t)}{dt}=f_\theta(z(t),t),\quad t\in[0,T].$$

Zamiast jawnie liczyć wielki jakobian mapy końcowej, używamy równania ciągłości gęstości:
$$\frac{d}{dt}\log p(z(t))=-\operatorname{tr}\left(\frac{\partial f_\theta}{\partial z}\right).$$

Ślad pochodnej pola wektorowego to dywergencja. Wzdłuż trajektorii całkujemy ją w czasie, aby skorygować log-gęstość. FFJORD (Free-form Jacobian of Reversible Dynamics) stosuje Hutchinson trace estimator: dla wektora szumu $\epsilon$ o $E[\epsilon\epsilon^T]=I$, $\operatorname{tr}(J)=E[\epsilon^T J\epsilon]$. Pozwala to oszacować ślad przez iloczyny Jacobian-vector, bez konstruowania całej macierzy.

Solver ODE jest częścią modelu: tolerancje i liczba kroków wpływają na koszt oraz dokładność. CNF ma elastyczną dynamikę, ale może wymagać wielu ewaluacji pola (NFE); backprop przez solver bywa pamięciożerny. FFJORD używa adjoint sensitivity, żeby zmniejszyć zużycie pamięci, kosztem dodatkowych obliczeń.


In [ ]:
# Mały, pedagogiczny CNF w 2D: stały krok Eulera i dokładny ślad.
# Nie jest to produkcyjny solver; pokazuje równania i znak korekty.
class VectorField(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3,hidden),nn.Tanh(),nn.Linear(hidden,hidden),nn.Tanh(),nn.Linear(hidden,2))
    def forward(self,z,t):
        time = torch.full((z.shape[0],1),float(t),device=z.device,dtype=z.dtype)
        return self.net(torch.cat([z,time],dim=-1))

def divergence_exact(field, z, t):
    # D=2: jawny ślad, z-gradient dla obu składowych.
    z = z.requires_grad_(True); dz = field(z,t); tr = 0.
    for d in range(z.shape[1]):
        grad = torch.autograd.grad(dz[:,d].sum(),z,create_graph=True,retain_graph=True)[0]
        tr = tr + grad[:,d]
    return dz, tr

def integrate_cnf(field,z0,steps=32):
    z=z0; delta_logp=torch.zeros(z.shape[0],device=z.device)
    dt=1.0/steps
    for i in range(steps):
        t=i/steps
        dz, div = divergence_exact(field,z,t)
        z = z + dt*dz
        # d log p(z(t))/dt = -div f
        delta_logp = delta_logp - dt*div
    return z, delta_logp

field=VectorField().to(device)
z0=torch.randn(16,2,device=device)
z1, correction=integrate_cnf(field,z0,steps=8)
print('Kształt stanu:', z1.shape, '| korekta log p (pierwsze 4):', correction[:4].detach())

## 4. Pełny likelihood z CNF

Przyjmijmy, że $z(0)=x$ to obserwacja, a solver ewoluuje do $z(T)$, którego gęstość bazowa znamy. Integracja daje:
$$\log p(x)=\log p(z(T))+\int_0^T \operatorname{div} f(z(t),t)\,dt.$$

W kodzie `delta_logp = -integral(div)` odpowiada zmianie log-gęstości trajektorii od początku do końca. Zatem $\log p(x)=\log p(z(T))-\text{delta\_logp}$. Znak wynika z równania ciągłości. To dobry punkt kontrolny: znak w CNF łatwo pomylić.

Wymiar 2 pozwala policzyć dokładny ślad. W wymiarze $D$ wymagałoby to $D$ gradientów dla każdej ewaluacji. Estymator Hutchinson bierze losowe $\epsilon$ i oblicza gradient skalaru $f(z)^T\epsilon$ względem $z$, a następnie iloczyn z $\epsilon$. Wysoka wariancja estimatorów może utrudnić optymalizację; wiele implementacji utrzymuje tę samą epsilon w całym ODE solve.


In [ ]:
def cnf_log_prob(field, x, steps=16):
    zT, delta = integrate_cnf(field, x, steps=steps)
    log_p_base = -0.5*(zT.square()+math.log(2*math.pi)).sum(-1)
    return log_p_base - delta

lp = cnf_log_prob(field,z0,steps=8)
print('CNF log_prob shape:',lp.shape,'| finite:',torch.isfinite(lp).all().item())

def divergence_hutchinson(field,z,t,eps=None):
    """Nieobciążony estymator śladu; eps może być Rademacher lub Gauss."""
    if eps is None: eps=torch.empty_like(z).bernoulli_(0.5).mul_(2).sub_(1)
    z=z.requires_grad_(True); fz=field(z,t)
    jte=torch.autograd.grad((fz*eps).sum(),z,create_graph=True)[0]
    return (jte*eps).sum(-1)

ztest=torch.randn(5,2,device=device)
print('Dokładny div:',divergence_exact(field,ztest,0.2)[1].detach())
print('Hutchinson div:',divergence_hutchinson(field,ztest,0.2).detach())

## 5. Porównanie rodzin i wybór modelu

| Rodzina | Główna idea | Zmiana log-gęstości | Typowe mocne strony | Koszt / ograniczenia |
|---|---|---|---|---|
| RealNVP | Affine coupling, maski i permutacje | Suma log-skali | Prosta implementacja, dokładny likelihood | Wymaga wielu warstw; mixing ograniczony |
| Glow | ActNorm + uczone 1×1 conv + coupling | Suma skali i $HW\log|\det W|$ | Lepsze mieszanie kanałów, wieloskalowe obrazy | Duże modele i kosztowna pamięć/obliczenia |
| FFJORD/CNF | Ciągła dynamika ODE | Integral dywergencji | Elastyczna trajektoria, free-form pole | Solver/NFE, koszt gradientów, błąd numeryczny |

Inne ważne linie rozwoju obejmują planar/radial flows (proste wariacyjne przepływy), autoregressive flows (MAF/IAF, różny kompromis szybkości gęstości i próbkowania), spline flows (bardziej elastyczna monotoniczna transformacja 1D) oraz residual flows. Nazwy „flow” obejmują różne rodziny warstw, ale wszędzie rachunek zmiany zmiennych pozostaje wspólną podstawą.

## Podsumowanie i ścieżka dalszej nauki

Glow pokazuje, że można uczyć także samą permutację kanałów, a ActNorm poprawia skalowanie aktywacji. FFJORD zamienia dyskretną listę warstw na ciągłą trajektorię i ślad jakobianu na całkę dywergencji. Wybór zależy od danych, wymaganego likelihood, budżetu obliczeń i tego, czy priorytetem jest szybkość próbkowania czy oceny gęstości.

Proponowana kolejność ćwiczeń: zmierz round-trip i logdet w RealNVP; dodaj ActNorm/1×1 conv; porównaj ślad dokładny z Hutchinson w 2D; zmniejsz krok solvera i obserwuj stabilność. Dla pełnych obrazów użyj właściwej wieloskalowej architektury, dequantyzacji i walidacji na ustalonym benchmarku.
